# 08 · Confirmatory analysis (protocol v1.4)
Six blocks of the 120-cell design (720 scenarios) x 5 packages x 2 modes x 2 repetitions with Qwen3-8B (14,400 runs), the H6
labels-synced arm, the block-1 ablation, and the Llama-3.1 replication on every block. The campaign runs with
`python harness/campaign.py` (resumable). **Interim-look rule (protocol §6):** until every phase is complete this notebook shows
run counts only; no outcome is computed or displayed.

In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "harness"), str(ROOT / "benchmark")]
import pandas as pd
from IPython.display import Markdown, Image, display
pd.set_option("display.max_colwidth", 120)
print("repository root:", ROOT)

repository root: C:\IEEE-RESEARCH-WORK


In [2]:
import campaign, analyze, run
status = pd.DataFrame([(n, *campaign.counts(campaign.OUT / f"{n}.jsonl")[1:], campaign.expected(kw))
                       for n, kw in [*campaign.reference_phases(), *campaign.llm_phases()]],
                      columns=["log", "done", "unresolved_errors", "planned"])
display(status)
COMPLETE = bool((status.done >= status.planned).all())
print("campaign complete:", COMPLETE, "- outcomes are analysed only when it is")

campaign complete: True - outcomes are analysed only when it is


,log,done,unresolved_errors,planned
0,ideal_confirmatory,2400,0,2400
1,ideal_confirmatory_b2,2400,0,2400
2,ideal_confirmatory_b3,2400,0,2400
3,ideal_confirmatory_b4,2400,0,2400
4,ideal_confirmatory_b5,2400,0,2400
5,ideal_confirmatory_b6,2400,0,2400
6,ideal_ablation_confirmatory,2160,0,2160
7,qwen3-8b_confirmatory,2400,0,2400
8,qwen3-8b_h6_confirmatory,24,0,24
9,qwen3-8b_confirmatory_b2,2400,1,2400


## Independent re-scoring (protocol §8)
Every confirmatory run is re-scored from its trace with the frozen scorer and compared field by field with the logged scores.

In [3]:
if COMPLETE:
    from env import Env
    from score import score_run
    splits = {json.loads(m.read_text())["items_sha256"]: m.parent for m in (ROOT / "benchmark/data").glob("*/MANIFEST.json")}
    items_of = {h: {i["item_id"]: i for i in map(json.loads, (d / "items.jsonl").read_text().splitlines())} for h, d in splits.items()}
    fields = ["gold_action", "reachable", "unreachable_reason", "refused", "substantive", "correct", "hedged", "sar", "supported",
              "supported_cond", "abcr", "exposed", "abcr_exposed", "blocked_attempts", "forbidden_read", "forbidden_cited", "disclosed",
              "deployed_violation", "refusal_correct", "over_restriction", "drift_exposed", "label"]
    n = mismatch = 0
    serving = {}  # protocol v1.4: one server version, model digest and clean harness commit per model over the whole campaign
    for p in sorted(campaign.OUT.glob("*.jsonl")):
        for r in map(json.loads, p.read_text().splitlines()):
            if "serving" in r and "error" not in r["result"]:  # completed records only; failed attempts carry no data (§6)
                serving.setdefault(r["model"], set()).add((r["serving"]["version"], r["serving"]["digest"], r["code"]))
            h = r["data_sha256"]["items"]
            env = Env(splits[h], r["stage"], r["mode"], r["profile"], r["seed"])
            env.trace = r["trace"]
            new = score_run(items_of[h][r["item_id"]], env, r["result"])
            n, mismatch = n + 1, mismatch + any(new[f] != r[f] for f in fields)
    print(f"re-scored {n} confirmatory runs; mismatches: {mismatch}")
    print("serving and code per model:", {k: sorted(v) for k, v in serving.items()})
    assert all(len(v) == 1 for v in serving.values()), "the server, model weights or harness changed during the campaign"
    assert not any(c.endswith("+dirty") for v in serving.values() for _, _, c in v), "runs from an uncommitted harness"
    assert mismatch == 0

re-scored 37674 confirmatory runs; mismatches: 0
serving and code per model: {'llama3.1-tools:8b': [('0.40.13', '528696a4274c', 'e90d92682ae3')], 'qwen3:8b': [('0.40.13', '500a1f067a9f', 'e90d92682ae3')]}


## Analyses
Primary: Qwen3 over all six blocks (scenario-cluster bootstrap stratified by block, Holm over the 9 contrasts, sign-flip check,
SESOI classes; H6 outside the family; per-block and design-cell sensitivities). Replication: Llama-3.1 (unadjusted). Block 1
alone (the split signed in v1.3a) as a sensitivity analysis.

In [4]:
if COMPLETE:
    import contextlib, io
    C, B = campaign.OUT, run.confirmatory_splits()
    log = lambda pat: [str(C / pat.format(b)) for b in B]
    jobs = {"analysis": [*log("qwen3-8b_{}.jsonl"), "--ideal", *log("ideal_{}.jsonl"), "--h6", *log("qwen3-8b_h6_{}.jsonl"),
                         "--ablation", str(C / "qwen3-8b_ablation_confirmatory.jsonl"),
                         "--ablation-ideal", str(C / "ideal_ablation_confirmatory.jsonl")],
            "analysis_llama": [*log("llama3.1-8b_{}.jsonl"), "--ideal", *log("ideal_{}.jsonl"), "--h6", *log("llama3.1-8b_h6_{}.jsonl"),
                               "--prefix", "Llama"],
            "analysis_block1": [str(C / "qwen3-8b_confirmatory.jsonl"), "--ideal", str(C / "ideal_confirmatory.jsonl"),
                                "--h6", str(C / "qwen3-8b_h6_confirmatory.jsonl"), "--prefix", "BlockOne"]}
    for out, args in jobs.items():
        sys.argv = ["analyze.py", *args, "--outdir", str(C / out)]
        with contextlib.redirect_stdout(io.StringIO()):
            analyze.main()
        print("wrote", (C / out).relative_to(ROOT))
    display(Markdown((C / "analysis/REPORT.md").read_text(encoding="utf-8")))

wrote runs\confirmatory\analysis
wrote runs\confirmatory\analysis_llama
wrote runs\confirmatory\analysis_block1


# Analysis of `qwen3-8b_confirmatory.jsonl`, `qwen3-8b_confirmatory_b2.jsonl`, `qwen3-8b_confirmatory_b3.jsonl`, `qwen3-8b_confirmatory_b4.jsonl`, `qwen3-8b_confirmatory_b5.jsonl`, `qwen3-8b_confirmatory_b6.jsonl`

Runs: 14400 (1 with unresolved infrastructure errors, excluded from inference); items: 720; model(s): ['qwen3:8b']; reference agent: observable-ideal

All intervals are item-cluster bootstrap 95% CIs (items weighted equally).


## Rates by stage x mode

| stage         | mode   |   runs | SAR                           | SuAR                          | ABCR                          | Accuracy                      | SuAR|answered                 | ABCR (all runs)               | Refusal correctness           | Over-restriction              | Stated-policy violation       | Hedged                        |   blocked |
|:--------------|:-------|-------:|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|----------:|
| development   | prompt |   1440 | 0.62 [0.59, 0.66] (576 items) | 0.59 [0.56, 0.63] (576 items) | 0.90 [0.87, 0.92] (667 items) | 0.61 [0.58, 0.65] (576 items) | 0.60 [0.57, 0.64] (571 items) | 0.90 [0.88, 0.92] (720 items) | 0.51 [0.42, 0.59] (144 items) | 0.02 [0.01, 0.03] (468 items) | 0.00 [0.00, 0.01] (720 items) | 0.01 [0.00, 0.01] (576 items) |         0 |
| development   | tool   |   1440 | 0.62 [0.59, 0.66] (576 items) | 0.59 [0.55, 0.63] (576 items) | 0.89 [0.87, 0.92] (667 items) | 0.61 [0.57, 0.65] (576 items) | 0.59 [0.56, 0.63] (574 items) | 0.90 [0.88, 0.92] (720 items) | 0.47 [0.39, 0.55] (144 items) | 0.02 [0.01, 0.03] (468 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.01] (576 items) |         0 |
| integration   | prompt |   1440 | 0.57 [0.53, 0.61] (576 items) | 0.54 [0.51, 0.58] (576 items) | 0.90 [0.87, 0.92] (664 items) | 0.56 [0.53, 0.60] (576 items) | 0.54 [0.51, 0.58] (576 items) | 0.90 [0.88, 0.92] (720 items) | 0.58 [0.50, 0.65] (144 items) | 0.00 [0.00, 0.01] (468 items) | 0.04 [0.03, 0.05] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| integration   | tool   |   1440 | 0.60 [0.56, 0.64] (576 items) | 0.57 [0.53, 0.60] (576 items) | 0.94 [0.92, 0.96] (664 items) | 0.59 [0.55, 0.62] (576 items) | 0.57 [0.53, 0.61] (576 items) | 0.95 [0.93, 0.96] (720 items) | 0.64 [0.56, 0.71] (144 items) | 0.00 [0.00, 0.01] (468 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| preproduction | prompt |   1439 | 0.79 [0.75, 0.82] (576 items) | 0.74 [0.70, 0.77] (576 items) | 0.78 [0.75, 0.80] (664 items) | 0.75 [0.72, 0.78] (576 items) | 0.74 [0.70, 0.77] (576 items) | 0.79 [0.77, 0.82] (720 items) | 0.06 [0.03, 0.08] (144 items) | 0.00 [0.00, 0.00] (504 items) | 0.30 [0.27, 0.33] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| preproduction | tool   |   1440 | 0.82 [0.79, 0.84] (576 items) | 0.76 [0.72, 0.79] (576 items) | 1.00 [1.00, 1.00] (664 items) | 0.79 [0.76, 0.82] (576 items) | 0.77 [0.73, 0.80] (570 items) | 1.00 [1.00, 1.00] (720 items) | 0.69 [0.61, 0.75] (144 items) | 0.00 [0.00, 0.00] (504 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| production    | prompt |   1440 | 0.91 [0.89, 0.93] (576 items) | 0.85 [0.82, 0.87] (576 items) | 0.80 [0.77, 0.83] (664 items) | 0.86 [0.83, 0.89] (576 items) | 0.85 [0.82, 0.87] (576 items) | 0.81 [0.79, 0.84] (720 items) | 0.10 [0.06, 0.15] (144 items) | 0.00 [0.00, 0.00] (576 items) | 0.19 [0.16, 0.21] (720 items) | 0.00 [0.00, 0.01] (576 items) |         0 |
| production    | tool   |   1440 | 0.94 [0.92, 0.96] (576 items) | 0.86 [0.83, 0.88] (576 items) | 1.00 [1.00, 1.00] (664 items) | 0.87 [0.84, 0.89] (576 items) | 0.86 [0.83, 0.88] (576 items) | 1.00 [1.00, 1.00] (720 items) | 0.69 [0.62, 0.75] (144 items) | 0.00 [0.00, 0.00] (576 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.01] (576 items) |         0 |
| postrelease   | prompt |   1440 | 0.84 [0.81, 0.87] (504 items) | 0.79 [0.76, 0.82] (504 items) | 0.67 [0.64, 0.70] (654 items) | 0.81 [0.78, 0.84] (504 items) | 0.79 [0.76, 0.82] (504 items) | 0.70 [0.67, 0.73] (720 items) | 0.04 [0.03, 0.06] (216 items) | 0.00 [0.00, 0.00] (504 items) | 0.30 [0.27, 0.33] (720 items) | 0.00 [0.00, 0.00] (504 items) |         0 |
| postrelease   | tool   |   1440 | 0.88 [0.86, 0.91] (504 items) | 0.82 [0.79, 0.85] (504 items) | 0.88 [0.86, 0.90] (654 items) | 0.85 [0.82, 0.87] (504 items) | 0.83 [0.79, 0.85] (504 items) | 0.89 [0.87, 0.91] (720 items) | 0.47 [0.41, 0.53] (216 items) | 0.00 [0.00, 0.00] (504 items) | 0.11 [0.09, 0.13] (720 items) | 0.00 [0.00, 0.00] (504 items) |         0 |


## Rates by profile

| profile    |   runs | SAR                          | SuAR                         | ABCR                          | Accuracy                     | SuAR|answered                | ABCR (all runs)               | Refusal correctness          | Over-restriction             | Stated-policy violation       | Hedged                       |   blocked |
|:-----------|-------:|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|----------:|
| compliance |   2399 | 0.77 [0.72, 0.81] (96 items) | 0.73 [0.69, 0.78] (96 items) | 0.89 [0.86, 0.92] (116 items) | 0.75 [0.71, 0.79] (96 items) | 0.73 [0.69, 0.78] (96 items) | 0.90 [0.87, 0.93] (120 items) | 0.30 [0.20, 0.40] (36 items) | 0.00 [0.00, 0.00] (96 items) | 0.09 [0.07, 0.11] (120 items) | 0.01 [0.00, 0.01] (96 items) |         0 |
| executive  |   2400 | 0.74 [0.71, 0.78] (96 items) | 0.69 [0.65, 0.74] (96 items) | 0.80 [0.75, 0.85] (102 items) | 0.71 [0.67, 0.76] (96 items) | 0.71 [0.67, 0.76] (96 items) | 0.85 [0.80, 0.89] (120 items) | 0.25 [0.17, 0.34] (36 items) | 0.02 [0.01, 0.03] (96 items) | 0.11 [0.08, 0.13] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| finance    |   2400 | 0.76 [0.72, 0.80] (96 items) | 0.71 [0.66, 0.76] (96 items) | 0.89 [0.85, 0.92] (117 items) | 0.72 [0.66, 0.77] (96 items) | 0.71 [0.66, 0.76] (96 items) | 0.89 [0.86, 0.92] (120 items) | 0.38 [0.25, 0.50] (36 items) | 0.00 [0.00, 0.00] (96 items) | 0.09 [0.07, 0.11] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| hr         |   2400 | 0.76 [0.71, 0.81] (96 items) | 0.71 [0.65, 0.76] (96 items) | 0.84 [0.79, 0.88] (112 items) | 0.72 [0.67, 0.77] (96 items) | 0.71 [0.66, 0.76] (96 items) | 0.86 [0.81, 0.90] (120 items) | 0.18 [0.12, 0.25] (36 items) | 0.00 [0.00, 0.00] (96 items) | 0.09 [0.07, 0.12] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| operations |   2400 | 0.74 [0.69, 0.79] (96 items) | 0.70 [0.64, 0.75] (96 items) | 0.91 [0.89, 0.94] (117 items) | 0.73 [0.67, 0.77] (96 items) | 0.70 [0.64, 0.75] (96 items) | 0.91 [0.89, 0.94] (120 items) | 0.37 [0.27, 0.48] (36 items) | 0.00 [0.00, 0.00] (96 items) | 0.09 [0.07, 0.11] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| sales      |   2400 | 0.75 [0.70, 0.79] (96 items) | 0.68 [0.63, 0.73] (96 items) | 0.89 [0.85, 0.92] (115 items) | 0.71 [0.66, 0.76] (96 items) | 0.68 [0.63, 0.74] (96 items) | 0.90 [0.87, 0.93] (120 items) | 0.31 [0.22, 0.40] (36 items) | 0.00 [0.00, 0.01] (96 items) | 0.09 [0.07, 0.11] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |


## Rates by condition

| condition           |   runs | SAR                           | SuAR                          | ABCR                          | Accuracy                      | SuAR|answered                 | ABCR (all runs)               | Refusal correctness           | Over-restriction              | Stated-policy violation       | Hedged                        |   blocked |
|:--------------------|-------:|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|----------:|
| authoritative       |   2880 | 0.85 [0.83, 0.87] (144 items) | 0.81 [0.77, 0.83] (144 items) | 0.95 [0.93, 0.98] (134 items) | 0.81 [0.78, 0.84] (144 items) | 0.81 [0.78, 0.84] (144 items) | 0.97 [0.96, 0.98] (144 items) | 0.00 [0.00, 0.00] (18 items)  | 0.01 [0.00, 0.02] (144 items) | 0.04 [0.03, 0.06] (144 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| distractor          |   2879 | 0.69 [0.63, 0.73] (144 items) | 0.63 [0.57, 0.67] (144 items) | 0.96 [0.94, 0.97] (132 items) | 0.67 [0.62, 0.72] (144 items) | 0.63 [0.57, 0.68] (144 items) | 0.97 [0.96, 0.98] (144 items) | 0.00 [0.00, 0.00] (18 items)  | 0.00 [0.00, 0.00] (144 items) | 0.04 [0.03, 0.05] (144 items) | 0.00 [0.00, 0.01] (144 items) |         0 |
| no_permitted_source |   2880 | -                             | -                             | 0.62 [0.59, 0.66] (144 items) | -                             | -                             | 0.62 [0.59, 0.66] (144 items) | 0.45 [0.40, 0.49] (144 items) | -                             | 0.25 [0.24, 0.27] (144 items) | -                             |         0 |
| prohibited          |   2880 | 0.81 [0.78, 0.84] (144 items) | 0.76 [0.73, 0.80] (144 items) | 0.90 [0.87, 0.92] (144 items) | 0.78 [0.74, 0.81] (144 items) | 0.77 [0.74, 0.80] (144 items) | 0.90 [0.87, 0.92] (144 items) | 0.00 [0.00, 0.00] (18 items)  | 0.00 [0.00, 0.01] (144 items) | 0.09 [0.07, 0.11] (144 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| stale               |   2880 | 0.67 [0.65, 0.70] (144 items) | 0.62 [0.59, 0.66] (144 items) | 0.95 [0.91, 0.97] (125 items) | 0.64 [0.60, 0.67] (144 items) | 0.63 [0.59, 0.66] (144 items) | 0.97 [0.96, 0.98] (144 items) | 0.00 [0.00, 0.00] (18 items)  | 0.00 [0.00, 0.01] (144 items) | 0.04 [0.03, 0.05] (144 items) | 0.00 [0.00, 0.01] (144 items) |         0 |


## Rates by task_class

| task_class   |   runs | SAR                           | SuAR                          | ABCR                          | Accuracy                      | SuAR|answered                 | ABCR (all runs)               | Refusal correctness           | Over-restriction              | Stated-policy violation       | Hedged                        |   blocked |
|:-------------|-------:|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|----------:|
| calculation  |   2880 | 0.67 [0.63, 0.71] (144 items) | 0.54 [0.49, 0.58] (144 items) | 0.88 [0.85, 0.91] (140 items) | 0.55 [0.51, 0.60] (144 items) | 0.54 [0.50, 0.59] (144 items) | 0.91 [0.88, 0.93] (144 items) | 0.00 [0.00, 0.00] (24 items)  | 0.01 [0.01, 0.02] (144 items) | 0.10 [0.08, 0.12] (144 items) | 0.01 [0.00, 0.01] (144 items) |         0 |
| comparison   |   2879 | 0.72 [0.68, 0.76] (144 items) | 0.67 [0.63, 0.71] (144 items) | 0.93 [0.90, 0.95] (140 items) | 0.73 [0.69, 0.76] (144 items) | 0.67 [0.63, 0.71] (144 items) | 0.94 [0.91, 0.96] (144 items) | 0.00 [0.00, 0.00] (24 items)  | 0.00 [0.00, 0.01] (144 items) | 0.06 [0.04, 0.07] (144 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| explanation  |   2880 | 0.86 [0.84, 0.88] (144 items) | 0.86 [0.83, 0.88] (144 items) | 1.00 [1.00, 1.00] (113 items) | 0.86 [0.83, 0.88] (144 items) | 0.86 [0.84, 0.88] (144 items) | 1.00 [1.00, 1.00] (144 items) | -                             | 0.00 [0.00, 0.00] (144 items) | 0.01 [0.01, 0.02] (144 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| lookup       |   2880 | 0.77 [0.73, 0.80] (144 items) | 0.76 [0.72, 0.80] (144 items) | 0.96 [0.94, 0.97] (142 items) | 0.76 [0.72, 0.80] (144 items) | 0.77 [0.73, 0.80] (144 items) | 0.96 [0.95, 0.98] (144 items) | 0.00 [0.00, 0.00] (24 items)  | 0.00 [0.00, 0.01] (144 items) | 0.05 [0.03, 0.06] (144 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| restricted   |   2880 | -                             | -                             | 0.62 [0.59, 0.66] (144 items) | -                             | -                             | 0.62 [0.59, 0.66] (144 items) | 0.45 [0.40, 0.49] (144 items) | -                             | 0.25 [0.24, 0.27] (144 items) | -                             |         0 |


## Rates by format

| format   |   runs | SAR                           | SuAR                          | ABCR                          | Accuracy                      | SuAR|answered                 | ABCR (all runs)               | Refusal correctness           | Over-restriction              | Stated-policy violation       | Hedged                        |   blocked |
|:---------|-------:|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|----------:|
| csv      |   4319 | 0.75 [0.72, 0.78] (216 items) | 0.69 [0.66, 0.73] (216 items) | 0.95 [0.93, 0.96] (211 items) | 0.71 [0.68, 0.75] (216 items) | 0.70 [0.66, 0.73] (216 items) | 0.96 [0.95, 0.97] (216 items) | 0.00 [0.00, 0.00] (30 items)  | 0.00 [0.00, 0.01] (216 items) | 0.05 [0.04, 0.06] (216 items) | 0.00 [0.00, 0.00] (216 items) |         0 |
| doc      |   5760 | 0.86 [0.84, 0.88] (144 items) | 0.86 [0.83, 0.88] (144 items) | 0.79 [0.76, 0.82] (257 items) | 0.86 [0.83, 0.88] (144 items) | 0.86 [0.84, 0.88] (144 items) | 0.81 [0.78, 0.84] (288 items) | 0.45 [0.40, 0.49] (144 items) | 0.00 [0.00, 0.00] (144 items) | 0.13 [0.12, 0.15] (288 items) | 0.00 [0.00, 0.00] (144 items) |         0 |
| json     |   4320 | 0.68 [0.65, 0.72] (216 items) | 0.62 [0.58, 0.65] (216 items) | 0.90 [0.87, 0.92] (211 items) | 0.65 [0.61, 0.68] (216 items) | 0.62 [0.58, 0.66] (216 items) | 0.91 [0.89, 0.93] (216 items) | 0.00 [0.00, 0.00] (42 items)  | 0.01 [0.01, 0.01] (216 items) | 0.08 [0.07, 0.10] (216 items) | 0.00 [0.00, 0.01] (216 items) |         0 |


## Rates by distractor_type

| distractor_type   |   runs | SAR                          | SuAR                         | ABCR                         | Accuracy                     | SuAR|answered                | ABCR (all runs)              | Refusal correctness         | Over-restriction             | Stated-policy violation      | Hedged                       |   blocked |
|:------------------|-------:|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|----------:|
| prior_period      |   1079 | 0.86 [0.82, 0.90] (54 items) | 0.74 [0.66, 0.80] (54 items) | 0.95 [0.91, 0.97] (54 items) | 0.74 [0.68, 0.80] (54 items) | 0.74 [0.66, 0.80] (54 items) | 0.96 [0.94, 0.98] (54 items) | 0.00 [0.00, 0.00] (9 items) | 0.00 [0.00, 0.00] (54 items) | 0.05 [0.03, 0.07] (54 items) | 0.00 [0.00, 0.00] (54 items) |         0 |
| sister_scope      |   1080 | 0.38 [0.31, 0.45] (54 items) | 0.34 [0.27, 0.42] (54 items) | 0.95 [0.92, 0.97] (52 items) | 0.46 [0.38, 0.54] (54 items) | 0.34 [0.27, 0.42] (54 items) | 0.96 [0.94, 0.98] (54 items) | 0.00 [0.00, 0.00] (9 items) | 0.00 [0.00, 0.01] (54 items) | 0.05 [0.03, 0.07] (54 items) | 0.01 [0.00, 0.02] (54 items) |         0 |


## Rates by prohibited_cue

|   prohibited_cue |   runs | SAR                          | SuAR                         | ABCR                         | Accuracy                     | SuAR|answered                | ABCR (all runs)              | Refusal correctness         | Over-restriction             | Stated-policy violation      | Hedged                       |   blocked |
|-----------------:|-------:|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|:----------------------------|:-----------------------------|:-----------------------------|:-----------------------------|----------:|
|                0 |   1440 | 0.78 [0.74, 0.83] (72 items) | 0.74 [0.68, 0.79] (72 items) | 0.87 [0.83, 0.92] (72 items) | 0.75 [0.69, 0.80] (72 items) | 0.74 [0.68, 0.79] (72 items) | 0.87 [0.83, 0.92] (72 items) | 0.00 [0.00, 0.00] (9 items) | 0.00 [0.00, 0.00] (72 items) | 0.10 [0.07, 0.13] (72 items) | 0.00 [0.00, 0.01] (72 items) |         0 |
|                1 |   1440 | 0.83 [0.80, 0.87] (72 items) | 0.79 [0.75, 0.84] (72 items) | 0.93 [0.90, 0.95] (72 items) | 0.80 [0.76, 0.84] (72 items) | 0.80 [0.76, 0.84] (72 items) | 0.93 [0.90, 0.95] (72 items) | 0.00 [0.00, 0.00] (9 items) | 0.01 [0.00, 0.01] (72 items) | 0.08 [0.06, 0.10] (72 items) | 0.00 [0.00, 0.00] (72 items) |         0 |


## Rates by stage x profile

| stage         | profile    |   runs | SAR                          | SuAR                         | ABCR                          | Accuracy                     | SuAR|answered                | ABCR (all runs)               | Refusal correctness          | Over-restriction             | Stated-policy violation       | Hedged                       |   blocked |
|:--------------|:-----------|-------:|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|----------:|
| development   | compliance |    480 | 0.64 [0.56, 0.73] (96 items) | 0.58 [0.49, 0.66] (96 items) | 0.91 [0.87, 0.96] (113 items) | 0.59 [0.51, 0.67] (96 items) | 0.58 [0.49, 0.66] (96 items) | 0.92 [0.87, 0.96] (120 items) | 0.51 [0.31, 0.71] (24 items) | 0.00 [0.00, 0.00] (78 items) | 0.00 [0.00, 0.00] (120 items) | 0.03 [0.01, 0.05] (96 items) |         0 |
| development   | executive  |    480 | 0.54 [0.45, 0.62] (96 items) | 0.52 [0.43, 0.60] (96 items) | 0.83 [0.75, 0.90] (99 items)  | 0.55 [0.47, 0.64] (96 items) | 0.58 [0.48, 0.66] (95 items) | 0.86 [0.80, 0.91] (120 items) | 0.48 [0.29, 0.67] (24 items) | 0.11 [0.06, 0.15] (78 items) | 0.00 [0.00, 0.01] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
| development   | finance    |    480 | 0.66 [0.57, 0.74] (96 items) | 0.63 [0.54, 0.71] (96 items) | 0.92 [0.87, 0.95] (115 items) | 0.64 [0.55, 0.72] (96 items) | 0.63 [0.54, 0.71] (96 items) | 0.92 [0.88, 0.96] (120 items) | 0.65 [0.49, 0.80] (24 items) | 0.00 [0.00, 0.01] (78 items) | 0.00 [0.00, 0.01] (120 items) | 0.01 [0.00, 0.01] (96 items) |         0 |
| development   | hr         |    480 | 0.65 [0.56, 0.73] (96 items) | 0.62 [0.52, 0.71] (96 items) | 0.83 [0.76, 0.89] (109 items) | 0.64 [0.55, 0.73] (96 items) | 0.62 [0.52, 0.71] (96 items) | 0.84 [0.78, 0.90] (120 items) | 0.10 [0.02, 0.21] (24 items) | 0.00 [0.00, 0.00] (78 items) | 0.00 [0.00, 0.00] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| development   | operations |    480 | 0.63 [0.53, 0.72] (96 items) | 0.61 [0.51, 0.70] (96 items) | 0.96 [0.93, 0.99] (117 items) | 0.63 [0.53, 0.71] (96 items) | 0.61 [0.51, 0.70] (96 items) | 0.96 [0.93, 0.99] (120 items) | 0.71 [0.54, 0.88] (24 items) | 0.00 [0.00, 0.00] (78 items) | 0.00 [0.00, 0.01] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| development   | sales      |    480 | 0.62 [0.54, 0.71] (96 items) | 0.59 [0.50, 0.68] (96 items) | 0.91 [0.86, 0.96] (114 items) | 0.60 [0.51, 0.69] (96 items) | 0.59 [0.50, 0.68] (96 items) | 0.92 [0.87, 0.96] (120 items) | 0.49 [0.31, 0.68] (24 items) | 0.00 [0.00, 0.00] (78 items) | 0.00 [0.00, 0.01] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | compliance |    480 | 0.61 [0.51, 0.69] (96 items) | 0.60 [0.51, 0.68] (96 items) | 0.98 [0.96, 0.99] (113 items) | 0.63 [0.55, 0.72] (96 items) | 0.60 [0.51, 0.69] (96 items) | 0.98 [0.96, 0.99] (120 items) | 0.77 [0.64, 0.90] (24 items) | 0.00 [0.00, 0.01] (78 items) | 0.02 [0.01, 0.04] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | executive  |    480 | 0.60 [0.50, 0.70] (96 items) | 0.54 [0.44, 0.63] (96 items) | 0.84 [0.77, 0.91] (99 items)  | 0.54 [0.44, 0.63] (96 items) | 0.54 [0.45, 0.63] (96 items) | 0.87 [0.81, 0.92] (120 items) | 0.47 [0.28, 0.65] (24 items) | 0.01 [0.00, 0.02] (78 items) | 0.01 [0.00, 0.02] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | finance    |    480 | 0.55 [0.46, 0.63] (96 items) | 0.51 [0.43, 0.60] (96 items) | 0.92 [0.87, 0.96] (112 items) | 0.53 [0.44, 0.61] (96 items) | 0.51 [0.43, 0.60] (96 items) | 0.92 [0.88, 0.96] (120 items) | 0.62 [0.46, 0.79] (24 items) | 0.00 [0.00, 0.01] (78 items) | 0.03 [0.01, 0.05] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | hr         |    480 | 0.59 [0.49, 0.67] (96 items) | 0.55 [0.46, 0.64] (96 items) | 0.85 [0.78, 0.91] (109 items) | 0.58 [0.49, 0.67] (96 items) | 0.56 [0.46, 0.64] (96 items) | 0.86 [0.80, 0.92] (120 items) | 0.26 [0.11, 0.41] (24 items) | 0.00 [0.00, 0.01] (78 items) | 0.01 [0.00, 0.03] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | operations |    480 | 0.57 [0.47, 0.66] (96 items) | 0.55 [0.45, 0.64] (96 items) | 0.97 [0.95, 0.99] (117 items) | 0.56 [0.46, 0.65] (96 items) | 0.55 [0.45, 0.64] (96 items) | 0.97 [0.96, 0.99] (120 items) | 0.83 [0.72, 0.94] (24 items) | 0.00 [0.00, 0.00] (78 items) | 0.02 [0.00, 0.04] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| integration   | sales      |    480 | 0.59 [0.51, 0.68] (96 items) | 0.57 [0.49, 0.66] (96 items) | 0.94 [0.91, 0.97] (114 items) | 0.61 [0.53, 0.69] (96 items) | 0.57 [0.49, 0.66] (96 items) | 0.94 [0.91, 0.97] (120 items) | 0.69 [0.54, 0.82] (24 items) | 0.00 [0.00, 0.01] (78 items) | 0.02 [0.01, 0.04] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| preproduction | compliance |    479 | 0.81 [0.73, 0.88] (96 items) | 0.78 [0.70, 0.85] (96 items) | 0.90 [0.86, 0.93] (113 items) | 0.81 [0.74, 0.87] (96 items) | 0.78 [0.70, 0.85] (96 items) | 0.90 [0.87, 0.94] (120 items) | 0.32 [0.23, 0.42] (24 items) | 0.00 [0.00, 0.00] (84 items) | 0.14 [0.10, 0.18] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
| preproduction | executive  |    480 | 0.82 [0.75, 0.88] (96 items) | 0.77 [0.69, 0.84] (96 items) | 0.84 [0.80, 0.88] (99 items)  | 0.79 [0.71, 0.86] (96 items) | 0.77 [0.69, 0.84] (96 items) | 0.86 [0.83, 0.90] (120 items) | 0.32 [0.23, 0.42] (24 items) | 0.00 [0.00, 0.00] (84 items) | 0.16 [0.13, 0.21] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| preproduction | finance    |    480 | 0.80 [0.72, 0.87] (96 items) | 0.75 [0.67, 0.83] (96 items) | 0.89 [0.85, 0.93] (112 items) | 0.76 [0.68, 0.83] (96 items) | 0.75 [0.67, 0.83] (96 items) | 0.90 [0.87, 0.93] (120 items) | 0.50 [0.40, 0.60] (24 items) | 0.00 [0.00, 0.00] (84 items) | 0.15 [0.11, 0.19] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| preproduction | hr         |    480 | 0.81 [0.73, 0.88] (96 items) | 0.75 [0.67, 0.83] (96 items) | 0.90 [0.86, 0.93] (109 items) | 0.77 [0.68, 0.84] (96 items) | 0.75 [0.67, 0.83] (96 items) | 0.91 [0.87, 0.94] (120 items) | 0.31 [0.23, 0.41] (24 items) | 0.00 [0.00, 0.00] (84 items) | 0.15 [0.11, 0.19] (120 items) | 0.01 [0.00, 0.01] (96 items) |         0 |
| preproduction | operations |    480 | 0.79 [0.71, 0.86] (96 items) | 0.74 [0.65, 0.82] (96 items) | 0.90 [0.86, 0.94] (117 items) | 0.77 [0.69, 0.84] (96 items) | 0.74 [0.65, 0.82] (96 items) | 0.90 [0.87, 0.94] (120 items) | 0.41 [0.32, 0.48] (24 items) | 0.00 [0.00, 0.00] (84 items) | 0.14 [0.11, 0.19] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| preproduction | sales      |    480 | 0.78 [0.70, 0.85] (96 items) | 0.69 [0.60, 0.77] (96 items) | 0.90 [0.87, 0.93] (114 items) | 0.72 [0.65, 0.80] (96 items) | 0.69 [0.61, 0.77] (96 items) | 0.90 [0.87, 0.94] (120 items) | 0.36 [0.28, 0.45] (24 items) | 0.01 [0.00, 0.02] (84 items) | 0.15 [0.10, 0.19] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| production    | compliance |    480 | 0.93 [0.88, 0.97] (96 items) | 0.89 [0.83, 0.94] (96 items) | 0.91 [0.88, 0.94] (113 items) | 0.89 [0.84, 0.94] (96 items) | 0.89 [0.83, 0.94] (96 items) | 0.92 [0.88, 0.95] (120 items) | 0.35 [0.26, 0.46] (24 items) | 0.00 [0.00, 0.00] (96 items) | 0.08 [0.05, 0.12] (120 items) | 0.01 [0.00, 0.02] (96 items) |         0 |
| production    | executive  |    480 | 0.92 [0.88, 0.95] (96 items) | 0.87 [0.82, 0.92] (96 items) | 0.84 [0.80, 0.88] (99 items)  | 0.88 [0.84, 0.92] (96 items) | 0.87 [0.82, 0.92] (96 items) | 0.87 [0.83, 0.91] (120 items) | 0.27 [0.19, 0.35] (24 items) | 0.00 [0.00, 0.00] (96 items) | 0.13 [0.09, 0.17] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
| production    | finance    |    480 | 0.93 [0.90, 0.97] (96 items) | 0.84 [0.78, 0.91] (96 items) | 0.92 [0.88, 0.95] (112 items) | 0.84 [0.78, 0.91] (96 items) | 0.84 [0.78, 0.91] (96 items) | 0.92 [0.89, 0.95] (120 items) | 0.54 [0.41, 0.68] (24 items) | 0.00 [0.00, 0.00] (96 items) | 0.08 [0.05, 0.11] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| production    | hr         |    480 | 0.93 [0.88, 0.97] (96 items) | 0.85 [0.78, 0.91] (96 items) | 0.89 [0.86, 0.93] (109 items) | 0.85 [0.78, 0.91] (96 items) | 0.85 [0.78, 0.91] (96 items) | 0.90 [0.86, 0.94] (120 items) | 0.33 [0.25, 0.41] (24 items) | 0.00 [0.00, 0.00] (96 items) | 0.10 [0.06, 0.14] (120 items) | 0.01 [0.00, 0.01] (96 items) |         0 |
| production    | operations |    480 | 0.92 [0.86, 0.96] (96 items) | 0.85 [0.78, 0.91] (96 items) | 0.92 [0.88, 0.95] (117 items) | 0.87 [0.81, 0.92] (96 items) | 0.85 [0.78, 0.91] (96 items) | 0.92 [0.89, 0.95] (120 items) | 0.48 [0.39, 0.58] (24 items) | 0.00 [0.00, 0.00] (96 items) | 0.08 [0.05, 0.11] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
| production    | sales      |    480 | 0.91 [0.85, 0.96] (96 items) | 0.80 [0.73, 0.87] (96 items) | 0.91 [0.88, 0.94] (114 items) | 0.84 [0.77, 0.90] (96 items) | 0.80 [0.73, 0.87] (96 items) | 0.91 [0.88, 0.94] (120 items) | 0.41 [0.31, 0.49] (24 items) | 0.01 [0.00, 0.01] (96 items) | 0.09 [0.06, 0.12] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
| postrelease   | compliance |    480 | 0.89 [0.83, 0.94] (84 items) | 0.85 [0.78, 0.91] (84 items) | 0.79 [0.72, 0.85] (114 items) | 0.88 [0.81, 0.93] (84 items) | 0.85 [0.79, 0.92] (84 items) | 0.80 [0.72, 0.86] (120 items) | 0.19 [0.12, 0.26] (36 items) | 0.01 [0.00, 0.02] (84 items) | 0.21 [0.14, 0.28] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |
| postrelease   | executive  |    480 | 0.89 [0.85, 0.93] (84 items) | 0.83 [0.77, 0.89] (84 items) | 0.72 [0.65, 0.78] (99 items)  | 0.85 [0.79, 0.90] (84 items) | 0.83 [0.77, 0.89] (84 items) | 0.77 [0.71, 0.83] (120 items) | 0.24 [0.17, 0.32] (36 items) | 0.00 [0.00, 0.00] (84 items) | 0.23 [0.17, 0.29] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |
| postrelease   | finance    |    480 | 0.88 [0.83, 0.93] (84 items) | 0.84 [0.76, 0.90] (84 items) | 0.79 [0.73, 0.85] (112 items) | 0.84 [0.77, 0.90] (84 items) | 0.84 [0.76, 0.90] (84 items) | 0.81 [0.75, 0.86] (120 items) | 0.35 [0.24, 0.46] (36 items) | 0.00 [0.00, 0.01] (84 items) | 0.19 [0.14, 0.25] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |
| postrelease   | hr         |    480 | 0.84 [0.78, 0.90] (84 items) | 0.79 [0.71, 0.85] (84 items) | 0.77 [0.70, 0.82] (110 items) | 0.79 [0.71, 0.85] (84 items) | 0.79 [0.71, 0.85] (84 items) | 0.79 [0.72, 0.84] (120 items) | 0.24 [0.16, 0.32] (36 items) | 0.00 [0.00, 0.00] (84 items) | 0.21 [0.16, 0.28] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |
| postrelease   | operations |    480 | 0.82 [0.74, 0.89] (84 items) | 0.77 [0.69, 0.85] (84 items) | 0.78 [0.72, 0.84] (108 items) | 0.82 [0.74, 0.89] (84 items) | 0.77 [0.69, 0.86] (84 items) | 0.81 [0.74, 0.86] (120 items) | 0.25 [0.17, 0.33] (36 items) | 0.00 [0.00, 0.01] (84 items) | 0.19 [0.14, 0.26] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |
| postrelease   | sales      |    480 | 0.85 [0.78, 0.91] (84 items) | 0.77 [0.68, 0.84] (84 items) | 0.80 [0.74, 0.86] (111 items) | 0.80 [0.72, 0.87] (84 items) | 0.77 [0.68, 0.84] (84 items) | 0.81 [0.76, 0.87] (120 items) | 0.27 [0.18, 0.36] (36 items) | 0.00 [0.00, 0.00] (84 items) | 0.19 [0.13, 0.24] (120 items) | 0.00 [0.00, 0.00] (84 items) |         0 |


## Rates by block

|   block |   runs | SAR                          | SuAR                         | ABCR                          | Accuracy                     | SuAR|answered                | ABCR (all runs)               | Refusal correctness          | Over-restriction             | Stated-policy violation       | Hedged                       |   blocked |
|--------:|-------:|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|:-----------------------------|:------------------------------|:-----------------------------|----------:|
|       1 |   2400 | 0.77 [0.72, 0.81] (96 items) | 0.73 [0.68, 0.77] (96 items) | 0.86 [0.82, 0.90] (115 items) | 0.74 [0.70, 0.79] (96 items) | 0.73 [0.68, 0.78] (96 items) | 0.88 [0.84, 0.91] (120 items) | 0.34 [0.24, 0.43] (36 items) | 0.01 [0.00, 0.01] (96 items) | 0.10 [0.08, 0.12] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
|       2 |   2399 | 0.77 [0.72, 0.81] (96 items) | 0.72 [0.66, 0.76] (96 items) | 0.89 [0.85, 0.92] (115 items) | 0.73 [0.68, 0.78] (96 items) | 0.72 [0.67, 0.77] (96 items) | 0.90 [0.87, 0.93] (120 items) | 0.28 [0.18, 0.38] (36 items) | 0.01 [0.00, 0.01] (96 items) | 0.08 [0.06, 0.10] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
|       3 |   2400 | 0.76 [0.72, 0.80] (96 items) | 0.71 [0.66, 0.76] (96 items) | 0.87 [0.84, 0.91] (110 items) | 0.73 [0.68, 0.77] (96 items) | 0.71 [0.66, 0.77] (96 items) | 0.89 [0.85, 0.92] (120 items) | 0.27 [0.17, 0.37] (36 items) | 0.00 [0.00, 0.01] (96 items) | 0.09 [0.07, 0.12] (120 items) | 0.00 [0.00, 0.01] (96 items) |         0 |
|       4 |   2400 | 0.75 [0.70, 0.79] (96 items) | 0.69 [0.64, 0.74] (96 items) | 0.87 [0.83, 0.91] (114 items) | 0.72 [0.67, 0.76] (96 items) | 0.70 [0.65, 0.75] (96 items) | 0.89 [0.86, 0.92] (120 items) | 0.28 [0.19, 0.38] (36 items) | 0.01 [0.00, 0.01] (96 items) | 0.09 [0.07, 0.11] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
|       5 |   2400 | 0.74 [0.69, 0.79] (96 items) | 0.70 [0.65, 0.75] (96 items) | 0.87 [0.83, 0.91] (112 items) | 0.72 [0.67, 0.77] (96 items) | 0.70 [0.65, 0.75] (96 items) | 0.88 [0.85, 0.92] (120 items) | 0.27 [0.17, 0.36] (36 items) | 0.01 [0.00, 0.01] (96 items) | 0.09 [0.07, 0.12] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |
|       6 |   2400 | 0.74 [0.69, 0.79] (96 items) | 0.68 [0.63, 0.74] (96 items) | 0.86 [0.82, 0.90] (113 items) | 0.71 [0.65, 0.76] (96 items) | 0.69 [0.63, 0.74] (96 items) | 0.87 [0.83, 0.91] (120 items) | 0.36 [0.26, 0.46] (36 items) | 0.00 [0.00, 0.00] (96 items) | 0.10 [0.08, 0.13] (120 items) | 0.00 [0.00, 0.00] (96 items) |         0 |


## Observable-ideal reference agent by stage (what each package's faults allow; it obeys the stated policy)

| stage         |   runs | SAR                           | SuAR                          | ABCR                          | Accuracy                      | SuAR|answered                 | ABCR (all runs)               | Refusal correctness           | Over-restriction              | Stated-policy violation       | Hedged                        |   blocked |
|:--------------|-------:|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|:------------------------------|----------:|
| development   |   2880 | 0.73 [0.70, 0.76] (576 items) | 0.73 [0.70, 0.76] (576 items) | 1.00 [1.00, 1.00] (667 items) | 0.73 [0.70, 0.76] (576 items) | 0.73 [0.70, 0.76] (576 items) | 1.00 [1.00, 1.00] (720 items) | 1.00 [1.00, 1.00] (144 items) | 0.00 [0.00, 0.00] (468 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| integration   |   2880 | 0.73 [0.70, 0.76] (576 items) | 0.73 [0.70, 0.76] (576 items) | 1.00 [1.00, 1.00] (664 items) | 0.73 [0.70, 0.76] (576 items) | 0.73 [0.70, 0.76] (576 items) | 1.00 [1.00, 1.00] (720 items) | 1.00 [1.00, 1.00] (144 items) | 0.00 [0.00, 0.00] (468 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| preproduction |   2880 | 0.88 [0.85, 0.90] (576 items) | 0.88 [0.85, 0.90] (576 items) | 1.00 [1.00, 1.00] (664 items) | 0.88 [0.85, 0.90] (576 items) | 1.00 [1.00, 1.00] (504 items) | 1.00 [1.00, 1.00] (720 items) | 1.00 [1.00, 1.00] (144 items) | 0.00 [0.00, 0.00] (504 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| production    |   2880 | 1.00 [1.00, 1.00] (576 items) | 1.00 [1.00, 1.00] (576 items) | 1.00 [1.00, 1.00] (664 items) | 1.00 [1.00, 1.00] (576 items) | 1.00 [1.00, 1.00] (576 items) | 1.00 [1.00, 1.00] (720 items) | 1.00 [1.00, 1.00] (144 items) | 0.00 [0.00, 0.00] (576 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (576 items) |         0 |
| postrelease   |   2880 | 1.00 [1.00, 1.00] (504 items) | 1.00 [1.00, 1.00] (504 items) | 1.00 [1.00, 1.00] (654 items) | 1.00 [1.00, 1.00] (504 items) | 1.00 [1.00, 1.00] (504 items) | 1.00 [1.00, 1.00] (720 items) | 1.00 [1.00, 1.00] (216 items) | 0.00 [0.00, 0.00] (504 items) | 0.00 [0.00, 0.00] (720 items) | 0.00 [0.00, 0.00] (504 items) |         0 |


## Preregistered contrasts (b - a). faultfree = agent difference on items the reference agent solves in both packages (primary H1/H2a/b); drift = H2c over items exposed through the permission drift (fixed before treatment); exposed = the same over every post-release-exposed item (secondary); did = v1.1 difference-in-differences (secondary); costpkg = H3b by package; verdict uses the expected sign and Holm p

| family       | hyp        | outcome          | factor   | b           | a          | restrict                            | method     |   expected | estimate                |   items |      p |   p_holm | verdict    |
|:-------------|:-----------|:-----------------|:---------|:------------|:-----------|:------------------------------------|:-----------|-----------:|:------------------------|--------:|-------:|---------:|:-----------|
| confirmatory | H1         | sar              | stage    | development | production | all                                 | faultfree  |         -1 | -0.050 [-0.077, -0.024] |     397 | 0.0005 |   0.0045 | supported  |
| confirmatory | H1         | sar              | stage    | integration | production | all                                 | faultfree  |         -1 | -0.090 [-0.120, -0.060] |     397 | 0.0005 |   0.0045 | supported  |
| confirmatory | H1         | supported        | stage    | development | production | all                                 | faultfree  |         -1 | -0.037 [-0.069, -0.004] |     397 | 0.0285 |   0.0285 | supported  |
| confirmatory | H1         | supported        | stage    | integration | production | all                                 | faultfree  |         -1 | -0.062 [-0.096, -0.028] |     397 | 0.0005 |   0.0045 | supported  |
| confirmatory | H2         | correct_ans      | stage    | postrelease | production | all                                 | faultfree  |         -1 | -0.041 [-0.062, -0.020] |     504 | 0.0015 |   0.0045 | supported  |
| confirmatory | H2         | sar              | stage    | postrelease | production | all                                 | faultfree  |         -1 | -0.067 [-0.088, -0.046] |     504 | 0.0005 |   0.0045 | supported  |
| confirmatory | H2         | abcr_exposed     | stage    | postrelease | production | all                                 | drift      |         -1 | -0.255 [-0.303, -0.208] |     285 | 0.0005 |   0.0045 | supported  |
| confirmatory | H3         | abcr_exposed     | mode     | tool        | prompt     | development,integration,postrelease | raw        |          1 | +0.082 [+0.070, +0.095] |     679 | 0.0005 |   0.0045 | supported  |
| confirmatory | H3         | correct_ans      | mode     | tool        | prompt     | all                                 | raw        |          0 | +0.019 [+0.011, +0.028] |     576 | 0.0005 |   0.0045 | difference |
| descriptive  | did        | sar              | stage    | development | production | all                                 | did        |         -1 | -0.030 [-0.051, -0.010] |     576 | 0.0045 | nan      | nan        |
| descriptive  | did        | sar              | stage    | integration | production | all                                 | did        |         -1 | -0.069 [-0.092, -0.047] |     576 | 0.0005 | nan      | nan        |
| descriptive  | did        | supported        | stage    | development | production | all                                 | did        |         -1 | +0.010 [-0.017, +0.038] |     576 | 0.484  | nan      | nan        |
| descriptive  | did        | supported        | stage    | integration | production | all                                 | did        |         -1 | -0.025 [-0.052, +0.003] |     576 | 0.08   | nan      | nan        |
| descriptive  | did        | correct_ans      | stage    | postrelease | production | all                                 | did        |         -1 | -0.041 [-0.062, -0.020] |     504 | 0.0015 | nan      | nan        |
| descriptive  | did        | sar              | stage    | postrelease | production | all                                 | did        |         -1 | -0.067 [-0.088, -0.046] |     504 | 0.0005 | nan      | nan        |
| descriptive  | exposedall | abcr_exposed     | stage    | postrelease | production | all                                 | exposed    |         -1 | -0.122 [-0.146, -0.098] |     654 | 0.0005 | nan      | nan        |
| descriptive  | costpkg    | correct_ans      | mode     | tool        | prompt     | development                         | raw        |          0 | -0.004 [-0.017, +0.009] |     576 | 0.604  | nan      | nan        |
| descriptive  | costpkg    | correct_ans      | mode     | tool        | prompt     | integration                         | raw        |          0 | +0.022 [+0.006, +0.037] |     576 | 0.0065 | nan      | nan        |
| descriptive  | costpkg    | correct_ans      | mode     | tool        | prompt     | preproduction                       | raw        |          0 | +0.037 [+0.018, +0.056] |     576 | 0.0005 | nan      | nan        |
| descriptive  | costpkg    | correct_ans      | mode     | tool        | prompt     | production                          | raw        |          0 | +0.005 [-0.011, +0.022] |     576 | 0.573  | nan      | nan        |
| descriptive  | costpkg    | correct_ans      | mode     | tool        | prompt     | postrelease                         | raw        |          0 | +0.036 [+0.019, +0.055] |     504 | 0.0005 | nan      | nan        |
| descriptive  | total      | sar              | stage    | development | production | all                                 | raw        |         -1 | -0.301 [-0.339, -0.264] |     576 | 0.0005 | nan      | nan        |
| descriptive  | total      | supported        | stage    | development | production | all                                 | raw        |         -1 | -0.261 [-0.299, -0.223] |     576 | 0.0005 | nan      | nan        |
| descriptive  | total      | correct_ans      | stage    | postrelease | production | all                                 | raw        |         -1 | -0.041 [-0.062, -0.020] |     504 | 0.0015 | nan      | nan        |
| descriptive  | invariant  | abcr_exposed     | mode     | tool        | prompt     | production,preproduction            | raw        |          1 | +0.212 [+0.184, +0.242] |     664 | 0.0005 | nan      | nan        |
| descriptive  | cost       | sar              | mode     | tool        | prompt     | all                                 | raw        |          0 | +0.025 [+0.017, +0.035] |     576 | 0.0005 | nan      | nan        |
| descriptive  | cost       | over_restriction | mode     | tool        | prompt     | all                                 | raw        |          0 | +0.000 [-0.002, +0.003] |     576 | 0.649  | nan      | nan        |
| descriptive  | owndrift   | abcr_exposed     | stage    | postrelease | production | all                                 | owndrift   |         -1 | -0.903 [-0.955, -0.844] |      72 | 0.0005 | nan      | nan        |
| descriptive  | reachdrift | abcr_exposed     | stage    | postrelease | production | all                                 | reachdrift |         -1 | -0.036 [-0.059, -0.018] |     213 | 0.0005 | nan      | nan        |
| descriptive  | covered    | abcr_covered     | mode     | tool        | prompt     | development,integration,postrelease | raw        |          1 | +0.087 [+0.076, +0.099] |     679 | 0.0005 | nan      | nan        |
| descriptive  | open       | abcr_open        | mode     | tool        | prompt     | development,integration,postrelease | raw        |          0 | -0.003 [-0.008, +0.001] |     679 | 0.161  | nan      | nan        |


## Sensitivity: design cell as the cluster (blocks are instances of one 120-cell design)

| hyp   | outcome      | b           | method    |   expected |    diff |      lo |       hi |      p |   items |   nonzero |   p_signflip |   p_holm | verdict    |
|:------|:-------------|:------------|:----------|-----------:|--------:|--------:|---------:|-------:|--------:|----------:|-------------:|---------:|:-----------|
| H1    | sar          | development | faultfree |         -1 | -0.0659 | -0.116  | -0.0188  | 0.0065 |      89 |        35 |     0.0102   |   0.013  | supported  |
| H1    | sar          | integration | faultfree |         -1 | -0.108  | -0.156  | -0.0619  | 0.0005 |      89 |        50 |     2e-05    |   0.0045 | supported  |
| H1    | supported    | development | faultfree |         -1 | -0.0473 | -0.101  |  0.00215 | 0.064  |      89 |        51 |     0.083    |   0.064  | null       |
| H1    | supported    | integration | faultfree |         -1 | -0.0841 | -0.135  | -0.0355  | 0.0005 |      89 |        56 |     0.0015   |   0.0045 | supported  |
| H2    | correct_ans  | postrelease | faultfree |         -1 | -0.041  | -0.0632 | -0.0205  | 0.0005 |      96 |        45 |     0.000185 |   0.0045 | supported  |
| H2    | sar          | postrelease | faultfree |         -1 | -0.0635 | -0.0894 | -0.0405  | 0.0005 |      96 |        40 |     0        |   0.0045 | supported  |
| H2    | abcr_exposed | postrelease | drift     |         -1 | -0.251  | -0.315  | -0.191   | 0.0005 |      85 |        49 |     0        |   0.0045 | supported  |
| H3    | abcr_exposed | tool        | raw       |          1 |  0.0778 |  0.0555 |  0.102   | 0.0005 |     120 |        50 |     0        |   0.0045 | supported  |
| H3    | correct_ans  | tool        | raw       |          0 |  0.0192 |  0.0075 |  0.032   | 0.0005 |      96 |        63 |     0.00146  |   0.0045 | difference |

Verdicts that change: ['H1 supported development']


## H6 (v1.4, pilot-derived, outside the Holm family): own re-classified dataset read, prompt-only, deny list by identifier with the label lagging minus labels synced: +0.104 [+0.056, +0.160], p = 0.000, sign-flip p = 0.000, 72 scenarios (13 non-zero); verdict supported (supported, compatible with >= SESOI); read rate lagging 0.89, synced 0.78


## Stated-denied sources read, by how the deny reaches the agent (v1.4, descriptive)

| route      | description                                                                                | mode   |   rate |   lo |   hi |   scenarios |   runs |
|:-----------|:-------------------------------------------------------------------------------------------|:-------|-------:|-----:|-----:|------------:|-------:|
| Own        | identifier deny, label lags; targeted (post-release own re-classified dataset)             | prompt |   0.94 | 0.89 | 0.99 |          72 |    144 |
| Own        | identifier deny, label lags; targeted (post-release own re-classified dataset)             | tool   |   0.94 | 0.88 | 0.99 |          72 |    144 |
| Uat        | identifier deny, label permits; targeted (pre-production erroneous deny)                   | prompt |   0.97 | 0.92 | 1.00 |          72 |    144 |
| Uat        | identifier deny, label permits; targeted (pre-production erroneous deny)                   | tool   |   0.00 | 0.00 | 0.00 |          72 |    144 |
| Restricted | class deny on the label; targeted, no permitted source (production restricted questions)   | prompt |   0.79 | 0.72 | 0.85 |         144 |    288 |
| Restricted | class deny on the label; targeted, no permitted source (production restricted questions)   | tool   |   0.00 | 0.00 | 0.00 |         144 |    288 |
| Prohibited | class deny on the label; a permitted source also answers (production prohibited condition) | prompt |   0.13 | 0.09 | 0.17 |         144 |    288 |
| Prohibited | class deny on the label; a permitted source also answers (production prohibited condition) | tool   |   0.00 | 0.00 | 0.00 |         144 |    288 |


## H5: accuracy-only vs source/boundary-aware release gate (thresholds: accuracy .80; SuAR .80; ABCR .95)

| stage         | mode   |   accuracy |   SuAR |   ABCR | accuracy_gate   | aware_gate   | disagree   | driver    |   p_disagree |
|:--------------|:-------|-----------:|-------:|-------:|:----------------|:-------------|:-----------|:----------|-------------:|
| development   | prompt |      0.611 |  0.592 |  0.895 | hold            | hold         | False      |           |       0      |
| development   | tool   |      0.607 |  0.586 |  0.894 | hold            | hold         | False      |           |       0      |
| integration   | prompt |      0.563 |  0.543 |  0.897 | hold            | hold         | False      |           |       0      |
| integration   | tool   |      0.585 |  0.565 |  0.941 | hold            | hold         | False      |           |       0      |
| postrelease   | prompt |      0.811 |  0.791 |  0.67  | promote         | hold         | True       | SuAR+ABCR |       0.7355 |
| postrelease   | tool   |      0.846 |  0.823 |  0.881 | promote         | hold         | True       | ABCR      |       0.999  |
| preproduction | prompt |      0.749 |  0.736 |  0.777 | hold            | hold         | False      |           |       0.0015 |
| preproduction | tool   |      0.786 |  0.757 |  1     | hold            | hold         | False      |           |       0.2095 |
| production    | prompt |      0.86  |  0.845 |  0.798 | promote         | hold         | True       | ABCR      |       1      |
| production    | tool   |      0.865 |  0.855 |  1     | promote         | promote      | False      |           |       0      |

P(unsupported | correct) = 0.036 [0.026, 0.049] over 564 items (213 of 8153 correct runs unsupported; 33 correct runs are chance-correct, i.e. no received source yields the answer; 180 of the 8120 evidenced correct runs are unsupported); Kendall tau(accuracy, SuAR) across configurations = 1.00. Item bootstrap of the gate rule: P(>=1 disagreement) = 1.00, driven by SuAR 0.63, by ABCR 1.00; p_disagree = per-configuration share of resamples.


Sensitivity (v1.3a): a gate on task success over all questions (correct answers and correct refusals) instead of answer accuracy disagrees with the aware gate on 0 configurations.

| stage         | mode   |   accuracy |   SuAR |   ABCR | accuracy_gate   | aware_gate   | disagree   | driver   |   p_disagree |
|:--------------|:-------|-----------:|-------:|-------:|:----------------|:-------------|:-----------|:---------|-------------:|
| development   | prompt |      0.59  |  0.592 |  0.895 | hold            | hold         | False      |          |        0     |
| development   | tool   |      0.58  |  0.586 |  0.894 | hold            | hold         | False      |          |        0     |
| integration   | prompt |      0.567 |  0.543 |  0.897 | hold            | hold         | False      |          |        0     |
| integration   | tool   |      0.595 |  0.565 |  0.941 | hold            | hold         | False      |          |        0     |
| postrelease   | prompt |      0.581 |  0.791 |  0.67  | hold            | hold         | False      |          |        0     |
| postrelease   | tool   |      0.733 |  0.823 |  0.881 | hold            | hold         | False      |          |        0     |
| preproduction | prompt |      0.61  |  0.736 |  0.777 | hold            | hold         | False      |          |        0     |
| preproduction | tool   |      0.767 |  0.757 |  1     | hold            | hold         | False      |          |        0.013 |
| production    | prompt |      0.709 |  0.845 |  0.798 | hold            | hold         | False      |          |        0     |
| production    | tool   |      0.831 |  0.855 |  1     | promote         | promote      | False      |          |        0     |


## Leave-one-component-out ablation (descriptive; ablation repetitions only). diff = paired item-level difference from production; ideal_diff = the reference agent's (injected faults); ff = agent difference on items the reference agent solves in both (agent-attributable); nonzero = items that moved

| configuration                      | outcome   |    rate |    diff |      lo |      hi |   nonzero |   ideal_diff |      ff |   ff_lo |   ff_hi |   ff_items |
|:-----------------------------------|:----------|--------:|--------:|--------:|--------:|----------:|-------------:|--------:|--------:|--------:|-----------:|
| production                         | SAR       |   +0.94 | +nan    | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| production                         | SuAR      |   +0.88 | +nan    | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| production                         | Accuracy  |   +0.90 | +nan    | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| production                         | ABCR      |   +0.90 | +nan    | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| prod + connectors from integration | SAR       |   +0.92 |   -0.03 |   -0.06 |   +0.00 |     +6.00 |        +0.00 |   -0.03 |   -0.06 |   +0.00 |     +96.00 |
| prod + connectors from integration | SuAR      |   +0.85 |   -0.03 |   -0.07 |   +0.02 |    +11.00 |        +0.00 |   -0.03 |   -0.07 |   +0.02 |     +96.00 |
| prod + connectors from integration | Accuracy  |   +0.87 |   -0.03 |   -0.08 |   +0.02 |    +11.00 |        +0.00 |   -0.03 |   -0.08 |   +0.02 |     +96.00 |
| prod + connectors from integration | ABCR      |   +0.89 |   -0.01 |   -0.03 |   +0.01 |     +4.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + metadata from development   | SAR       |   +0.80 |   -0.14 |   -0.22 |   -0.07 |    +18.00 |        -0.10 |   -0.04 |   -0.09 |   +0.01 |     +86.00 |
| prod + metadata from development   | SuAR      |   +0.75 |   -0.12 |   -0.21 |   -0.05 |    +21.00 |        -0.10 |   -0.03 |   -0.10 |   +0.02 |     +86.00 |
| prod + metadata from development   | Accuracy  |   +0.77 |   -0.13 |   -0.21 |   -0.05 |    +22.00 |        -0.10 |   -0.04 |   -0.10 |   +0.02 |     +86.00 |
| prod + metadata from development   | ABCR      |   +0.93 |   +0.04 |   +0.01 |   +0.07 |    +12.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + metadata from integration   | SAR       |   +0.81 |   -0.14 |   -0.22 |   -0.06 |    +20.00 |        -0.10 |   -0.03 |   -0.09 |   +0.02 |     +86.00 |
| prod + metadata from integration   | SuAR      |   +0.76 |   -0.12 |   -0.21 |   -0.04 |    +23.00 |        -0.10 |   -0.03 |   -0.10 |   +0.04 |     +86.00 |
| prod + metadata from integration   | Accuracy  |   +0.78 |   -0.12 |   -0.21 |   -0.04 |    +22.00 |        -0.10 |   -0.03 |   -0.10 |   +0.04 |     +86.00 |
| prod + metadata from integration   | ABCR      |   +0.90 |   +0.00 |   -0.02 |   +0.03 |     +7.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + policy from development     | SAR       |   +0.94 |   -0.01 |   -0.04 |   +0.02 |     +4.00 |        +0.00 |   -0.01 |   -0.04 |   +0.02 |     +96.00 |
| prod + policy from development     | SuAR      |   +0.89 |   +0.01 |   -0.02 |   +0.04 |     +8.00 |        +0.00 |   +0.01 |   -0.02 |   +0.04 |     +96.00 |
| prod + policy from development     | Accuracy  |   +0.91 |   +0.01 |   -0.02 |   +0.03 |     +7.00 |        +0.00 |   +0.01 |   -0.02 |   +0.03 |     +96.00 |
| prod + policy from development     | ABCR      |   +0.84 |   -0.05 |   -0.10 |   -0.01 |    +20.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + policy from integration     | SAR       |   +0.93 |   -0.02 |   -0.06 |   +0.02 |     +7.00 |        +0.00 |   -0.02 |   -0.06 |   +0.02 |     +96.00 |
| prod + policy from integration     | SuAR      |   +0.89 |   +0.01 |   -0.03 |   +0.05 |    +10.00 |        +0.00 |   +0.01 |   -0.03 |   +0.05 |     +96.00 |
| prod + policy from integration     | Accuracy  |   +0.90 |   -0.01 |   -0.05 |   +0.03 |     +9.00 |        +0.00 |   -0.01 |   -0.05 |   +0.03 |     +96.00 |
| prod + policy from integration     | ABCR      |   +0.85 |   -0.05 |   -0.09 |   -0.01 |    +13.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + prompt from development     | SAR       |   +0.94 |   +0.00 |   -0.03 |   +0.03 |     +7.00 |        +0.00 |   +0.00 |   -0.03 |   +0.03 |     +96.00 |
| prod + prompt from development     | SuAR      |   +0.89 |   +0.02 |   -0.03 |   +0.06 |    +15.00 |        +0.00 |   +0.02 |   -0.03 |   +0.06 |     +96.00 |
| prod + prompt from development     | Accuracy  |   +0.91 |   +0.01 |   -0.04 |   +0.05 |    +11.00 |        +0.00 |   +0.01 |   -0.04 |   +0.05 |     +96.00 |
| prod + prompt from development     | ABCR      |   +0.91 |   +0.01 |   -0.01 |   +0.04 |     +7.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + prompt from integration     | SAR       |   +0.96 |   +0.02 |   -0.01 |   +0.04 |     +5.00 |        +0.00 |   +0.02 |   -0.01 |   +0.04 |     +96.00 |
| prod + prompt from integration     | SuAR      |   +0.91 |   +0.04 |   +0.00 |   +0.07 |    +13.00 |        +0.00 |   +0.04 |   +0.00 |   +0.07 |     +96.00 |
| prod + prompt from integration     | Accuracy  |   +0.92 |   +0.02 |   -0.01 |   +0.05 |    +10.00 |        +0.00 |   +0.02 |   -0.01 |   +0.05 |     +96.00 |
| prod + prompt from integration     | ABCR      |   +0.93 |   +0.04 |   +0.01 |   +0.06 |     +8.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + snapshot from development   | SAR       |   +0.75 |   -0.19 |   -0.27 |   -0.12 |    +20.00 |        -0.19 |   -0.03 |   -0.06 |   +0.00 |     +78.00 |
| prod + snapshot from development   | SuAR      |   +0.70 |   -0.17 |   -0.25 |   -0.10 |    +24.00 |        -0.19 |   -0.01 |   -0.05 |   +0.02 |     +78.00 |
| prod + snapshot from development   | Accuracy  |   +0.72 |   -0.18 |   -0.26 |   -0.10 |    +26.00 |        -0.19 |   -0.02 |   -0.06 |   +0.02 |     +78.00 |
| prod + snapshot from development   | ABCR      |   +0.88 |   -0.02 |   -0.05 |   +0.00 |     +9.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| prod + snapshot from integration   | SAR       |   +0.76 |   -0.18 |   -0.27 |   -0.10 |    +21.00 |        -0.19 |   +0.01 |   -0.01 |   +0.03 |     +78.00 |
| prod + snapshot from integration   | SuAR      |   +0.71 |   -0.16 |   -0.25 |   -0.08 |    +25.00 |        -0.19 |   +0.03 |   -0.01 |   +0.07 |     +78.00 |
| prod + snapshot from integration   | Accuracy  |   +0.72 |   -0.18 |   -0.26 |   -0.09 |    +24.00 |        -0.19 |   +0.01 |   -0.03 |   +0.04 |     +78.00 |
| prod + snapshot from integration   | ABCR      |   +0.89 |   -0.01 |   -0.03 |   +0.01 |     +4.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| development                        | SAR       |   +0.65 |   -0.29 |   -0.39 |   -0.20 |    +33.00 |        -0.28 |   -0.04 |   -0.09 |   +0.01 |     +69.00 |
| development                        | SuAR      |   +0.62 |   -0.25 |   -0.35 |   -0.15 |    +39.00 |        -0.28 |   -0.01 |   -0.09 |   +0.07 |     +69.00 |
| development                        | Accuracy  |   +0.66 |   -0.24 |   -0.35 |   -0.14 |    +37.00 |        -0.28 |   +0.01 |   -0.07 |   +0.09 |     +69.00 |
| development                        | ABCR      |   +0.91 |   +0.01 |   -0.03 |   +0.06 |    +22.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| integration                        | SAR       |   +0.64 |   -0.30 |   -0.40 |   -0.20 |    +36.00 |        -0.25 |   -0.07 |   -0.15 |   +0.00 |     +72.00 |
| integration                        | SuAR      |   +0.63 |   -0.24 |   -0.35 |   -0.14 |    +41.00 |        -0.25 |   -0.01 |   -0.10 |   +0.07 |     +72.00 |
| integration                        | Accuracy  |   +0.65 |   -0.26 |   -0.36 |   -0.15 |    +39.00 |        -0.25 |   -0.03 |   -0.11 |   +0.06 |     +72.00 |
| integration                        | ABCR      |   +0.91 |   +0.01 |   -0.04 |   +0.05 |    +20.00 |        +0.00 | +nan    | +nan    | +nan    |    +nan    |
| residual (development)             | SAR       | +nan    |   +0.05 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| residual (development)             | SuAR      | +nan    |   +0.02 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| residual (development)             | Accuracy  | +nan    |   +0.05 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| residual (integration)             | SAR       | +nan    |   +0.04 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| residual (integration)             | SuAR      | +nan    |   +0.02 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |
| residual (integration)             | Accuracy  | +nan    |   +0.06 | +nan    | +nan    |   +nan    |      +nan    | +nan    | +nan    | +nan    |    +nan    |


## Failure taxonomy (primary label counts)

|                             |   hallucination |   hedged_answer |   incomplete |   missed_refusal |   ok |   ok_refusal |   policy_violation |   reasoning_error |   safe_abstention |   stale_source |   stated_policy_violation |   unwarranted_refusal |   wrong_source |
|:----------------------------|----------------:|----------------:|-------------:|-----------------:|-----:|-------------:|-------------------:|------------------:|------------------:|---------------:|--------------------------:|----------------------:|---------------:|
| ('development', 'prompt')   |               0 |               6 |            0 |               33 |  680 |          146 |                140 |                35 |                 3 |            343 |                         0 |                    17 |             37 |
| ('development', 'tool')     |               0 |               2 |            0 |               36 |  675 |          136 |                141 |                41 |                 1 |            353 |                         0 |                    17 |             38 |
| ('integration', 'prompt')   |               2 |               0 |            0 |               30 |  625 |          167 |                137 |                31 |                 0 |            383 |                         0 |                     1 |             64 |
| ('integration', 'tool')     |               0 |               0 |            0 |               37 |  651 |          183 |                 79 |                38 |                 0 |            389 |                         0 |                     4 |             59 |
| ('preproduction', 'prompt') |               0 |               1 |            0 |               32 |  845 |           16 |                296 |                54 |                 0 |              0 |                       132 |                     1 |             62 |
| ('preproduction', 'tool')   |               0 |               2 |            4 |               89 |  872 |          198 |                  0 |                68 |                20 |             37 |                         0 |                     2 |            148 |
| ('production', 'prompt')    |               0 |               3 |            0 |               31 |  969 |           30 |                268 |                71 |                 0 |              1 |                         0 |                     1 |             66 |
| ('production', 'tool')      |               1 |               3 |            3 |               86 |  985 |          199 |                  0 |                96 |                 0 |              2 |                         0 |                     1 |             64 |
| ('postrelease', 'prompt')   |               0 |               0 |            0 |               42 |  790 |           19 |                431 |                52 |                 0 |             57 |                         1 |                     1 |             47 |
| ('postrelease', 'tool')     |               0 |               0 |            0 |               87 |  824 |          203 |                155 |                59 |                 0 |             64 |                         0 |                     2 |             46 |


## Failure taxonomy, agent-attributable (runs whose seed-matched reference-agent run succeeded)

|                             |   hallucination |   hedged_answer |   incomplete |   missed_refusal |   ok |   ok_refusal |   policy_violation |   reasoning_error |   stale_source |   stated_policy_violation |   unwarranted_refusal |   wrong_source |
|:----------------------------|----------------:|----------------:|-------------:|-----------------:|-----:|-------------:|-------------------:|------------------:|---------------:|--------------------------:|----------------------:|---------------:|
| ('development', 'prompt')   |               0 |               2 |            0 |               33 |  679 |          146 |                134 |                35 |             52 |                         0 |                    14 |             33 |
| ('development', 'tool')     |               0 |               1 |            0 |               36 |  675 |          136 |                136 |                41 |             57 |                         0 |                    14 |             32 |
| ('integration', 'prompt')   |               2 |               0 |            0 |               30 |  625 |          167 |                135 |                31 |             75 |                         0 |                     1 |             62 |
| ('integration', 'tool')     |               0 |               0 |            0 |               37 |  651 |          183 |                 77 |                38 |             81 |                         0 |                     4 |             57 |
| ('preproduction', 'prompt') |               0 |               1 |            0 |               32 |  845 |           16 |                284 |                54 |              0 |                         5 |                     1 |             57 |
| ('preproduction', 'tool')   |               0 |               2 |            1 |               89 |  872 |          198 |                  0 |                68 |              2 |                         0 |                     2 |             62 |
| ('production', 'prompt')    |               0 |               3 |            0 |               31 |  969 |           30 |                268 |                71 |              1 |                         0 |                     1 |             66 |
| ('production', 'tool')      |               1 |               3 |            3 |               86 |  985 |          199 |                  0 |                96 |              2 |                         0 |                     1 |             64 |
| ('postrelease', 'prompt')   |               0 |               0 |            0 |               42 |  790 |           19 |                431 |                52 |             57 |                         1 |                     1 |             47 |
| ('postrelease', 'tool')     |               0 |               0 |            0 |               87 |  824 |          203 |                155 |                59 |             64 |                         0 |                     2 |             46 |


## Accuracy and refusal by evidence reachability (injected faults vs agent behaviour; chance = credited answers that no received source yields)

|                                          |   runs |   accuracy |   refused |   chance |
|:-----------------------------------------|-------:|-----------:|----------:|---------:|
| ('development', 'missing_from_snapshot') |    432 |      0.002 |     0.009 |        1 |
| ('development', 'reachable')             |   1872 |      0.749 |     0.018 |        0 |
| ('integration', 'old_version_served')    |    432 |      0.009 |     0     |        4 |
| ('integration', 'reachable')             |   1872 |      0.705 |     0.003 |        7 |
| ('postrelease', 'reachable')             |   2016 |      0.828 |     0.002 |        0 |
| ('preproduction', 'policy_denies')       |    288 |      0.056 |     0.069 |       16 |
| ('preproduction', 'reachable')           |   2015 |      0.87  |     0.001 |        3 |
| ('production', 'reachable')              |   2304 |      0.863 |     0.001 |        2 |


## Harness diagnostics

assistant turns 51737; tool calls recovered from plain text 5495; runs never calling submit_answer 20; infrastructure errors 1; blocked attempts 0; stated-policy violations 1344


## Secondary models: TOTAL package and mode effects (mostly injected faults; not the agent-attributable estimand). GEE odds ratios vs production / prompt-only; VB mixed-logit intervals in models.json are approximate

- **SAR** (n=11231): development OR 0.13 [0.10, 0.17]; integration OR 0.11 [0.09, 0.15]; postrelease OR 0.49 [0.39, 0.61]; preproduction OR 0.33 [0.26, 0.42]; tool OR 1.16 [1.10, 1.23]
- **SuAR** (n=11231): development OR 0.24 [0.19, 0.30]; integration OR 0.21 [0.17, 0.26]; postrelease OR 0.70 [0.60, 0.82]; preproduction OR 0.51 [0.44, 0.60]; tool OR 1.09 [1.04, 1.14]
- **ABCR** (n=13251): development OR 0.93 [0.72, 1.19]; integration OR 1.38 [1.04, 1.82]; postrelease OR 0.26 [0.21, 0.33]; preproduction OR 0.86 [0.78, 0.94]; tool OR 6.33 [5.31, 7.55]
- **Accuracy** (n=11231): development OR 0.24 [0.19, 0.30]; integration OR 0.20 [0.16, 0.25]; postrelease OR 0.74 [0.63, 0.87]; preproduction OR 0.52 [0.44, 0.61]; tool OR 1.11 [1.06, 1.16]